In [8]:
"""
NYC Taxi Analytics Platform - Silver Layer Ingestion
====================================================

This module implements production-grade Silver layer tranformations:
1. Schema standardization for Green and Yellow taxi datasets
2. Unified fact_taxi_trips creation
3. SCD Type 2 dimension table (dim_taxi_zone)
4. Comprehensive data quality checks
5. Incremental processing with idempotency

Author: Chirag Venkateshaiah
Last Updated: 2025-03-10
"""

from utils.spark_session import get_spark_session
from utils.hadoop_setup import complete_hadoop_setup

## Section 1: ENVIRONMENT SETUP

In [9]:
complete_hadoop_setup()
spark = get_spark_session()

from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, lit, current_date, current_timestamp, monotonically_increasing_id,
    to_date, year, month, concat_ws, md5, coalesce, row_number, when,
    sum as spark_sum, count as spark_count, max as spark_max
)
from pyspark.sql.window import Window
from pyspark.sql.types import StringType, IntegerType, DoubleType
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional
import traceback

✔ HADOOP_HOME set to:, os.environ['HADOOP_HOME']
✔ Added to PATH: C:\hadoop\bin

 ✔ winutils.exe: True
 ✔ hadoop.dll: True

🎉 Setup complete!


## Section 2: CONFIGURATION

In [10]:
PROJECT_ROOT = Path(r"C:\Users\chira\Desktop\data_engineering\PySpark\nyc-taxi-analytics-platform")


# Bronze paths
BRONZE_ROOT_PATH = PROJECT_ROOT / "data" / "bronze" / "nyc_taxi"

# Silver paths
SILVER_BASE_PATH = PROJECT_ROOT / "data" / "silver" / "nyc_taxi"
SILVER_FACT_PATH = SILVER_BASE_PATH / "fact_taxi_trips"
SILVER_DIM_ZONE_PATH = SILVER_BASE_PATH / "dim_taxi_zone"
SILVER_STANDARDIZED_GREEN_PATH = SILVER_BASE_PATH / "green_standardized"
SILVER_STANDARDIZED_YELLOW_PATH = SILVER_BASE_PATH / "yellow_standardized"

# Lookup data path
LOOKUP_BASE_PATH = PROJECT_ROOT / "data" / "lookup"
TAXI_ZONE_LOOKUP_PATH = LOOKUP_BASE_PATH / "taxi_zone_lookup.csv"

# Batch ID
BATCH_ID = f"batch_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Processing state tracker
SILVER_STATE_PATH = str(PROJECT_ROOT / "data" / "silver" / "_processing_state")



## Section 3 - DATA QUALITY & VALIDATION FRAMEWORK

In [11]:
class DataQualityChecker:
    """Encapsulates data quality validation logic"""

    def __init__(self, spark_session):
        self.spark = spark_session
        self.quality_metrics = {}

    
    def validate_fact_table(self, df: DataFrame, dataset_name: str) -> Tuple[DataFrame, Dict]:
        """
        Apply data quality rules to fact table
        Returns: (cleaned_df, metrics_dict)
        """
        print(f"\n{'-'*60}")
        print(f"Data Quality Validation: {dataset_name}")
        print(f"{'-'*60}")

        initial_count = df.count()
        print(f" Initial row count: {initial_count:,}")


        # Rule 1: Remove records with null timestamps
        df_clean = df.filter(
            col("pickup_ts").isNotNull() &
            col("dropoff_ts").isNotNull()
        )
        after_timestamp_check = df_clean.count()
        dropped_null_ts = initial_count - after_timestamp_check
        print(f"   ✔ Dropped {dropped_null_ts:,} rows with null timestamps")


        # Rule 2: Remove invalid trip distance (negative or zero)
        df_clean = df_clean.filter(col("trip_distance") > 0)
        after_distance_check = df_clean.count()
        dropped_invalid_distance = after_distance_check - after_timestamp_check
        print(f"    ✔ Dropped {dropped_invalid_distance:,} rows with invalid trip_distance")


        # Rule 3: Remove negative fare amounts
        df_clean = df_clean.filter(col("fare_amount") >= 0)
        after_fare_check = df_clean.count()
        dropped_negative_fare = after_distance_check - after_fare_check
        print(f"    ✔ Dropped {dropped_negative_fare:,} rows with negative fare_amount")


        # Rule 4: Remove negative total amounts
        df_clean = df_clean.filter(col("total_amount") >= 0)
        after_total_check = df_clean.count()
        dropped_negative_total = after_fare_check - after_total_check
        print(f"    ✔ Dropped {dropped_negative_total:,} rows with negative total_amount")


        # Rule 5: Validate passenger count (1-9, treating 0 as 1)
        df_clean = df_clean.withColumn(
            "passenger_count",
            when(col("passenger_count").between(1, 9), col("passenger_count"))
            .otherwise(1) # Default to 1 if invalid
        )
        print(f"    ✔ Normalized passenger_count (0 -> 1, invalid -> 1)")


        # Rule 6: Ensure dropoff > pickup
        df_clean = df_clean.filter(col("dropoff_ts") > col("pickup_ts"))
        after_time_logic_check = df_clean.count()
        dropped_invalid_time = after_total_check - after_time_logic_check
        print(f" ✔ Dropped {dropped_invalid_distance:,} rows where dropoff <= pickup")


        final_count = df_clean.count()
        total_dropped = initial_count - final_count
        retention_rate = (final_count / initial_count * 100) if initial_count > 0 else 0


        print(f"\n  Final row count: {final_count:,}")
        print(f"    Total dropped: {total_dropped:,} ({100-retention_rate:.2f}%)")
        print(f"    Retention rate: {retention_rate:.2f}%")


        metrics = {
            "dataset": dataset_name,
            "initial_count": initial_count,
            "final_count": final_count,
            "dropped_null_timestamps": dropped_null_ts,
            "dropped_invalid_distance": dropped_invalid_distance,
            "dropped_negative_fare": dropped_negative_fare,
            "dropped_negative_total": dropped_negative_total,
            "dropped_invalid_time": dropped_invalid_time,
            "retention_rate": retention_rate
        }

        return df_clean, metrics
    

    def validate_dimension_table(self, df: DataFrame, dim_name: str) -> DataFrame:
        """Validate dimension table data quality"""
        print(f"\n{'-'*60}")
        print(f"Dimension Validation: {dim_name}")
        print(f"{'-'*60}")

        initial_count = df.count()
        print(f"    Initial row count: {initial_count:,}")


        # Remove rows with null natural keys
        df_clean = df.filter(col("LocationID").isNotNull())
        after_null_check = df_clean.count()
        dropped_null_keys = initial_count - after_null_check
        print(f"    ✔ Dropped {dropped_null_keys:,} rows with null LocationID")

        final_count = df_clean.count()
        print(f"    Final row count: {final_count}")

        return df_clean
    
    

